<a href="https://colab.research.google.com/github/Jules-Vatel/SSI_SPRING/blob/main/SSI_SPRING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

df = pd.read_csv("/content/Offer-Westort_April 28, 2026_08.08.csv")

df = df[-df['Treat_Party'].str.contains('ImportId',na=False)].copy()
df['Feeling_ThermR'] = pd.to_numeric(df['Q-FeelingThermR_1'], errors='coerce')
df['Feeling_ThermD'] = pd.to_numeric(df['Q-FeelingThermD_1'], errors='coerce')



In [31]:
#function for seven point party Id.
def seven_pt_pid(row):
  party = row['Q-PartyAffiliation']
  strength = row['Q-PolarAffiliation']
  lean = row['Q-MiddleAffiliation']

  if party == 'Democrat':
      return 1 if strength == 'Strong' else 2
  elif party == 'Republican':
      return 7 if strength == 'Strong' else 6
  elif party in ['Independent', 'No Preference', 'Other Party (Please Specify)']:
      if lean == 'Closer to Democratic Party':
          return 3
      elif lean == 'Closer to Republican Party':
          return 5
      elif lean == 'Neither':
          return 4
  return np.nan

df['PA']=df.apply(seven_pt_pid,axis=1)

In [28]:
#function that
def get_party(row):
    pa = row['Q-PartyAffiliation']
    if pa == 'Democrat':
        return 'Democrat'
    elif pa == 'Republican':
        return 'Republican'
    elif pa in ['Independent', 'No Preference', 'Other Party (Please Specify)']:
        ma = row.get('Q-MiddleAffiliation', np.nan)
        if ma == 'Closer to Democratic Party':
            return 'Democrat'
        elif ma == 'Closer to Republican Party':
            return 'Republican'
    return np.nan

df['Party'] = df.apply(get_party, axis=1)


In [29]:
df['ThermInP']  = np.where(df['Party'] == 'Democrat', df['Feeling_ThermD'], df['Feeling_ThermR'])
df['ThermOutP'] = np.where(df['Party'] == 'Democrat', df['Feeling_ThermR'], df['Feeling_ThermD'])
df['APpre'] = (df['ThermInP'] - df['ThermOutP']).abs()

comfort_map = {
    'Very uncomfortable': 1, 'Uncomfortable': 2, 'Somewhat uncomfortable': 3,
    'Neither comfortable nor uncomfortable': 4, 'Somewhat comfortable': 5,
    'Comfortable': 6, 'Very comfortable': 7, 'Very Comfortable': 7
}

for col in ['Q-InLawDist', 'Q-CloseFriendDist', 'Q-Social Distance 3']:
    df[col + '_n'] = df[col].map(comfort_map)
df['APpost'] = 8 - df[['Q-InLawDist_n', 'Q-CloseFriendDist_n',
                        'Q-Social Distance 3_n']].mean(axis=1)
reelect_map = {
    'Very unlikely': 1, 'Unlikely': 2, 'Somewhat unlikely': 3,
    'Neither likely nor unlikely': 4, 'Somewhat likely': 5,
    'Likely': 6, 'Very likely': 7
}
df['TB'] = df['Q-ReElection'].map(reelect_map)



In [40]:
df['Treat'] = (df['Treat_Party'] != 'Control').astype(int)
df['Pin']   = (df['Treat_Party'] == 'InParty').astype(int)
df['FE']    = (df['Treat_Frame'] == 'Electoral').astype(int)
df['FD']    = (df['Treat_Frame'] == 'Democracy').astype(int)

df['FE_x_Pin'] = df['FE'] * df['Pin']
df['FD_x_Pin'] = df['FD'] * df['Pin']

analysis_vars = ['APpost','TB','APpre','PA']
full = df.dropna(subset=analysis_vars).copy()
treated = full[full['Treat']==1].copy()

def run_ols(y,x_vars,data):
  Y=data[y]
  X=sm.add_constant(data[x_vars])
  return sm.OLS(Y,X).fit(cov_type='HC1')



In [75]:

#made a class to navigate the data better
#have access to each individual model,a 2d array containing all the models in a 2d array
# the class also has funciton that print the results from the regression direclty, I will add plotting functions and some other functions
class DATA_ANALYSIS:
  def __init__(self,F=full,T=treated):
    self.AP_1 = run_ols('APpost',['Treat','APpre','PA'],F)
    self.TB_1 = run_ols('TB',['Treat','APpre','PA'],F)
    self.AP_2 = run_ols('APpost',['Treat','APpre','PA'],T)
    self.TB_2 = run_ols('TB',['Pin','APpre','PA'],T)
    self.AP_3 = run_ols('APpost',['FE','FD','Pin','APpre','PA'],T)
    self.TB_3 = run_ols('TB',['FE','FD','Pin','APpre','PA'], T)
    self.AP_4 = run_ols('APpost',['FE','FD','Pin','FE_x_Pin','FD_x_Pin','APpre','PA'],T)
    self.TB_4 = run_ols('TB',['FE','FD','Pin','FE_x_Pin','FD_x_Pin','APpre','PA'],T)
    ftest_AP3 = AP_3.f_test("FE = FD")
    ftest_TB3 = TB_3.f_test("FE = FD")
    ftest_AP4 = AP_4.f_test("FE_x_Pin = 0, FD_x_Pin = 0")
    ftest_TB4 = TB_4.f_test("FE_x_Pin = 0, FD_x_Pin = 0")
    self.models=[[AP_1,AP_2,AP_3,AP_4],[TB_1,TB_2,TB_3,TB_4]]
    self.samples_full = len(F)
    self.sampes_treated = len(T)

  def print_results_all(self):
    labels=["Affective Polarization","Tolarance for Backsliding"]
    models = self.models
    for i in range(len(models)):
      for j in range(len(models[i])):
        print(f"Model#{i+1} Results ({labels[i]})")
        print(models[i][j].summary(),"\n")

  def print_results_all(self,model_num,id):
    models = self.models
    if(model_num>len(models[0])):
      raise ValueError(f"Not A Valid Model Number: There are {models[0].len()} models")
    n = None
    mn = model_num-1
    labels=["Affective Polarization","Tolarance for Backsliding"]
    if(id.upper()=="AP"):
      n = 0
    else:
      n = 1
    print(f"Model#{model_num} Results ({labels[n]})")
    print(models[n][mn].summary(),"\n")







In [77]:

results = DATAANALYSIS()


